# DigitExtractor ? YOLOv8 INT8 TFLite Training on Google Colab

Train one or more YOLO detector sizes in Google Colab, export **INT8-quantized TFLite** models, and compare their validation metrics in one run.

### What you need on Google Drive
Upload your DigitExtractor output folder. It must contain:

| Folder / File | Contents |
|---|---|
| `ROI_640/` | 640 ? 640 PNG images |
| `ROI_640_labels/` | Matching YOLO `.txt` label files |
| `yolo_classes.txt` *(auto-detected)* | One class name per line |
| `yolo_class_map.json` *(auto-detected)* | Exported by DigitExtractor |


## 1 · GPU Check

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU detected.\n'
        'Go to:  Runtime → Change runtime type → Hardware accelerator → GPU (T4)'
    )
print('✅ GPU available')
print(result.stdout.split('\n')[8])   # one-liner GPU summary

assert torch.cuda.is_available(), 'PyTorch cannot see CUDA — try restarting the runtime.'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3 · Install Dependencies

In [ ]:
%pip install ultralytics -q
import ultralytics
ultralytics.checks()


## 4 ? Configuration
> **Edit the values in this cell before running anything else.**


In [ ]:
import os

# ?? Path to your DigitExtractor output folder on Google Drive ?????????????????
# Must contain ROI_640/ and ROI_640_labels/ sub-folders.
DATASET_PATH = '/content/drive/MyDrive/DigitExtractor_Output/TrainingFiles'   # ? CHANGE THIS

# ?? Model selection ????????????????????????????????????????????????????????????
# YOLOv8 size keys: 'n' nano | 's' small | 'm' medium | 'l' large | 'x' xlarge
# Examples:
#   MODEL_SIZES = ['s', 'm']
#   MODEL_SIZES = ['s', 'm']
MODEL_SIZES = ['s', 'm']

# ?? Training hyper-parameters ?????????????????????????????????????????????????
EPOCHS     = 100          # total training epochs
BATCH_SIZE = 16           # 16 for T4, 32 for A100
VAL_SPLIT  = 0.15         # fraction of data held out for validation
SEED       = 42           # random seed for reproducible split

# ?? Where to save the final results on Google Drive ???????????????????????????
RESULTS_DRIVE_PATH = '/content/drive/MyDrive/DigitExtractor_Output/Outputs'

# ?? Validation / safety ???????????????????????????????????????????????????????
MODEL_SIZES = [size.strip().lower() for size in MODEL_SIZES]
valid_sizes = {'n', 's', 'm', 'l', 'x'}
if not MODEL_SIZES:
    raise ValueError('MODEL_SIZES must contain at least one YOLO size code.')
if any(size not in valid_sizes for size in MODEL_SIZES):
    raise ValueError(f'Unsupported model size in MODEL_SIZES: {MODEL_SIZES}')
if len(set(MODEL_SIZES)) != len(MODEL_SIZES):
    raise ValueError(f'MODEL_SIZES contains duplicates: {MODEL_SIZES}')

print('Configuration')
print(f'  Dataset      : {DATASET_PATH}')
print(f'  Model sizes  : {MODEL_SIZES}')
print(f'  Epochs       : {EPOCHS}')
print(f'  Batch        : {BATCH_SIZE}')
print(f'  Val split    : {int(VAL_SPLIT*100)} %')
print(f'  Results path : {RESULTS_DRIVE_PATH}')


## 5 · Validate Dataset

In [ ]:
import glob

IMAGES_DIR = os.path.join(DATASET_PATH, 'ROI_640')
LABELS_DIR = os.path.join(DATASET_PATH, 'ROI_640_labels')

if not os.path.isdir(IMAGES_DIR):
    raise FileNotFoundError(
        f'ROI_640/ not found at:\n  {IMAGES_DIR}\n'
        'Make sure DATASET_PATH points to your DigitExtractor output folder.'
    )
if not os.path.isdir(LABELS_DIR):
    raise FileNotFoundError(
        f'ROI_640_labels/ not found at:\n  {LABELS_DIR}\n'
        'Make sure DATASET_PATH points to your DigitExtractor output folder.'
    )

# collect all images and labels
all_imgs = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp'):
    all_imgs.extend(glob.glob(os.path.join(IMAGES_DIR, ext)))
all_lbls = glob.glob(os.path.join(LABELS_DIR, '*.txt'))

img_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in all_imgs}
lbl_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in all_lbls}

paired          = sorted(set(img_by_stem) & set(lbl_by_stem))
orphan_imgs     = set(img_by_stem) - set(lbl_by_stem)
orphan_lbls     = set(lbl_by_stem) - set(img_by_stem)

print(f'Images found       : {len(all_imgs)}')
print(f'Label files found  : {len(all_lbls)}')
print(f'Matched pairs      : {len(paired)}')
if orphan_imgs:
    print(f'⚠  Images without labels : {len(orphan_imgs)}  (will be skipped)')
if orphan_lbls:
    print(f'⚠  Labels without images : {len(orphan_lbls)}  (will be skipped)')

if not paired:
    raise ValueError(
        'No matched image-label pairs found.\n'
        'Check that your filenames match between ROI_640/ and ROI_640_labels/.'
    )

print(f'\n✅  {len(paired)} samples ready.')


## 6 · Grayscale Preprocessing

Convert every matched image to **3-channel grayscale** (`BGR → GRAY → BGR`) before the train/val split.
Keeping 3 channels ensures full compatibility with YOLOv8's default loader while training exclusively on luminance information — matching what the tester does at inference time.

The `img_by_stem` mapping is updated in-place so all subsequent cells automatically use the grayscale copies without any further changes.

In [ ]:
import cv2 as _cv2
import numpy as _np

GRAY_DIR = '/content/drive/MyDrive/DigitExtractor_Output/Grayscales'
os.makedirs(GRAY_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Robust image reader — three methods so nothing is silently skipped.
# cv2.imread() returns None for paths with non-ASCII or special characters,
# which is exactly what was causing most Drive images to be skipped before.
# ---------------------------------------------------------------------------
def _read_img(path):
    # 1) np.fromfile + imdecode: reads raw bytes first, bypasses path issues
    try:
        raw = _np.fromfile(path, dtype=_np.uint8)
        if raw.size > 0:
            img = _cv2.imdecode(raw, _cv2.IMREAD_COLOR)
            if img is not None:
                return img
    except Exception:
        pass
    # 2) PIL / Pillow: broadest format support, handles anything GIMP can open
    try:
        from PIL import Image
        pil = Image.open(path).convert('RGB')
        arr = _np.array(pil)
        return _cv2.cvtColor(arr, _cv2.COLOR_RGB2BGR)
    except Exception:
        pass
    # 3) plain cv2.imread as absolute last resort
    return _cv2.imread(path, _cv2.IMREAD_COLOR)

total    = len(img_by_stem)
_done    = 0
_skipped = 0
_failed  = []

print(f'Converting {total} images to 3-channel grayscale...')
print(f'Output  ->  {GRAY_DIR}')

for stem, src_path in list(img_by_stem.items()):
    # Always save as .png regardless of source format
    dst_name = os.path.splitext(os.path.basename(src_path))[0] + '.png'
    dst_path = os.path.join(GRAY_DIR, dst_name)

    # ── Skip if already converted ──────────────────────────────────────────
    if os.path.isfile(dst_path):
        img_by_stem[stem] = dst_path   # redirect to existing grayscale copy
        _skipped += 1
        continue

    img = _read_img(src_path)
    if img is None:
        _failed.append(src_path)
        continue

    gray     = _cv2.cvtColor(img, _cv2.COLOR_BGR2GRAY)
    gray_3ch = _cv2.cvtColor(gray, _cv2.COLOR_GRAY2BGR)

    # imencode + tofile: also path-safe for writing
    ok, buf = _cv2.imencode('.png', gray_3ch)
    if not ok:
        _failed.append(f'{src_path}  (encode failed)')
        continue

    buf.tofile(dst_path)
    img_by_stem[stem] = dst_path   # redirect: split cell will copy this path
    _done += 1

IMAGES_DIR = GRAY_DIR  # keep in sync with later references

print(f'Skipped   : {_skipped} / {total}  (already in GRAY_DIR)')
print(f'Converted : {_done} / {total - _skipped}')
if _failed:
    print(f'Failed    : {len(_failed)}')
    for p in _failed[:15]:
        print(f'  {p}')
    if len(_failed) > 15:
        print(f'  ... and {len(_failed) - 15} more')

# ── Hard check: every img_by_stem entry must now point to GRAY_DIR ───────────
_still_color = [p for p in img_by_stem.values() if GRAY_DIR not in p]
if _still_color:
    raise RuntimeError(
        f'{len(_still_color)} image(s) were NOT converted and still point to '
        f'the original ROI_640 directory.\n'
        f'Fix the failures above before continuing — otherwise training will '
        f'mix color and grayscale images.'
    )

print(f'All {len(img_by_stem)} entries in img_by_stem -> {GRAY_DIR}')
print('Training WILL use grayscale. Safe to run Split cell.')


## 7 · Detect Class List

In [ ]:
# Classes are fixed: 0-11 (digit_strip, digit_0…digit_9, digit_unreadable).
# Hardcoded here to avoid slow Google Drive file reads and label scanning.
class_names = [
    'digit_strip',
    'digit_0', 'digit_1', 'digit_2', 'digit_3', 'digit_4',
    'digit_5', 'digit_6', 'digit_7', 'digit_8', 'digit_9',
    'digit_unreadable',
]

NC = len(class_names)
print(f'✅  Using fixed class list ({NC} classes):')
for i, name in enumerate(class_names):
    print(f'  {i:2d}  {name}')


## 8 · Train / Val Split

In [ ]:
import random, shutil

random.seed(SEED)
stems = paired[:]
random.shuffle(stems)

n_val   = max(1, round(len(stems) * VAL_SPLIT))
n_train = len(stems) - n_val
val_set   = set(stems[:n_val])
train_set = set(stems[n_val:])

print(f'Train : {n_train}')
print(f'Val   : {n_val}')

# Build the directory tree expected by ultralytics
WORK_DIR = '/content/yolo_dataset'
for split in ('train', 'val'):
    os.makedirs(os.path.join(WORK_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR, 'labels', split), exist_ok=True)

def _copy(stem, split):
    shutil.copy2(img_by_stem[stem],
                 os.path.join(WORK_DIR, 'images', split,
                              os.path.basename(img_by_stem[stem])))
    shutil.copy2(lbl_by_stem[stem],
                 os.path.join(WORK_DIR, 'labels', split,
                              os.path.basename(lbl_by_stem[stem])))

for s in train_set: _copy(s, 'train')
for s in val_set:   _copy(s, 'val')

print(f'\n✅  Files copied to {WORK_DIR}')


## 9 · Create data.yaml

In [ ]:
import yaml

DATA_YAML = os.path.join(WORK_DIR, 'data.yaml')

cfg = {
    'path' : WORK_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : NC,
    'names': class_names,
}

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print('✅  data.yaml written:')
print(open(DATA_YAML).read())


## 10 ? Train, Evaluate, and Export
Train each requested YOLO size **sequentially** using the same prepared dataset and validation split. Each model gets its own run folder, validation pass, and **INT8 TFLite** export.


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# ── INT8 export is disabled (commented out) ───────────────────────────────────
# INT8 calibration runs the full val set through the model and takes too long.
# Re-enable the export block below if you need a quantized .tflite later.
#
# import shutil, random, yaml
# INT8_CALIB_MAX = 200
#
# def _make_calib_yaml(src_yaml_path, calib_dir, max_imgs):
#     with open(src_yaml_path) as f:
#         cfg = yaml.safe_load(f)
#     dataset_root = cfg.get('path', '')
#     val_rel      = cfg.get('val', 'images/val')
#     val_dir      = Path(dataset_root) / val_rel
#     all_imgs = []
#     for ext in ('*.png', '*.jpg', '*.jpeg', '*.bmp'):
#         all_imgs.extend(val_dir.glob(ext))
#     random.seed(42)
#     sample = random.sample(all_imgs, min(max_imgs, len(all_imgs)))
#     calib_img_dir = Path(calib_dir) / 'images' / 'val'
#     calib_img_dir.mkdir(parents=True, exist_ok=True)
#     for p in sample:
#         shutil.copy2(p, calib_img_dir / p.name)
#     calib_cfg = {
#         'path' : str(calib_dir),
#         'train': 'images/val',
#         'val'  : 'images/val',
#         'nc'   : cfg['nc'],
#         'names': cfg['names'],
#     }
#     calib_yaml = Path(calib_dir) / 'calib_data.yaml'
#     with open(calib_yaml, 'w') as f:
#         yaml.dump(calib_cfg, f, default_flow_style=False, sort_keys=False)
#     print(f'  Calibration dataset : {len(sample)} images  (capped at {max_imgs})')
#     return str(calib_yaml)


RUN_RESULTS = []

for model_size in MODEL_SIZES:
    run_name = f'digit_extractor_{model_size}'
    print('\n' + '=' * 88)
    print(f'Training YOLOv8{model_size}  ->  run name: {run_name}')
    print('=' * 88)

    trainer = YOLO(f'yolov8{model_size}.pt')
    trainer.train(
        data         = DATA_YAML,
        epochs       = EPOCHS,
        imgsz        = 640,
        batch        = BATCH_SIZE,
        project      = '/content/runs',
        name         = run_name,
        optimizer    = 'AdamW',
        lr0          = 0.001,
        lrf          = 0.01,
        weight_decay = 0.0005,
        warmup_epochs= 3,
        patience     = 20,
        save         = True,
        save_period  = 10,
        plots        = True,
        verbose      = True,
        device       = 0,
        exist_ok     = True,
    )

    run_dir = Path(trainer.trainer.save_dir)
    best_pt = run_dir / 'weights' / 'best.pt'
    last_pt = run_dir / 'weights' / 'last.pt'

    print(f'Training complete for YOLOv8{model_size}')
    print(f'  Run dir  : {run_dir}')
    print(f'  Best .pt : {best_pt}')
    print(f'  Last .pt : {last_pt}')

    best_model = YOLO(str(best_pt))
    metrics = best_model.val(data=DATA_YAML, imgsz=640, device=0)

    print('\nValidation Metrics')
    print(f'mAP@50      : {metrics.box.map50:.4f}')
    print(f'mAP@50-95   : {metrics.box.map:.4f}')
    print(f'Precision   : {metrics.box.mp:.4f}')
    print(f'Recall      : {metrics.box.mr:.4f}')

    export_warning = ''
    tflite_path    = ''
    # ── INT8 TFLite export disabled ───────────────────────────────────────────
    # Uncomment the block below to re-enable when needed.
    # try:
    #     calib_dir  = f'/content/calib_{model_size}'
    #     calib_yaml = _make_calib_yaml(DATA_YAML, calib_dir, INT8_CALIB_MAX)
    #     exported_path = best_model.export(
    #         format='tflite',
    #         imgsz=640,
    #         int8=True,
    #         data=calib_yaml,
    #     )
    #     if exported_path:
    #         tflite_path = str(Path(exported_path))
    #         print(f'INT8 TFLite export: {tflite_path}')
    # except Exception as exc:
    #     export_warning = str(exc)
    #     print(f'TFLite export warning: {export_warning}')
    print('ℹ️  INT8 TFLite export skipped (disabled).')

    RUN_RESULTS.append({
        'model_size'    : model_size,
        'run_name'      : run_name,
        'run_dir'       : str(run_dir),
        'best_pt'       : str(best_pt),
        'last_pt'       : str(last_pt),
        'tflite_path'   : tflite_path,
        'export_warning': export_warning,
        'map50'         : float(metrics.box.map50),
        'map5095'       : float(metrics.box.map),
        'precision'     : float(metrics.box.mp),
        'recall'        : float(metrics.box.mr),
    })

print('\nDone training all requested model sizes.')


## 11 ? Compare Runs
Review every trained size in one summary table so you can compare `small` vs `medium` without repeating dataset preparation.


In [ ]:
if not RUN_RESULTS:
    raise RuntimeError('RUN_RESULTS is empty. Run the training cell first.')

headers = ['Size', 'mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Best .pt', 'INT8 .tflite']
rows = []
for result in RUN_RESULTS:
    rows.append([
        result['model_size'],
        f"{result['map50']:.4f}",
        f"{result['map5095']:.4f}",
        f"{result['precision']:.4f}",
        f"{result['recall']:.4f}",
        result['best_pt'],
        result['tflite_path'] or '(export failed)',
    ])

col_widths = [max(len(str(item)) for item in [header] + [row[idx] for row in rows]) for idx, header in enumerate(headers)]

def fmt_row(values):
    return ' | '.join(str(value).ljust(col_widths[idx]) for idx, value in enumerate(values))

print(fmt_row(headers))
print('-+-'.join('-' * width for width in col_widths))
for row in rows:
    print(fmt_row(row))

best_map50 = max(RUN_RESULTS, key=lambda item: item['map50'])
print('\nBest mAP@50 run:')
print(f"  YOLOv8{best_map50['model_size']}  ->  {best_map50['map50']:.4f}")


## 12 ? Save Results to Google Drive
Copy every run folder to Google Drive. If the exported TFLite file lives outside the run folder, copy it alongside the run artifacts too.


In [ ]:
import os
import shutil
from pathlib import Path

os.makedirs(RESULTS_DRIVE_PATH, exist_ok=True)

for result in RUN_RESULTS:
    run_dir = Path(result['run_dir'])
    dest = Path(RESULTS_DRIVE_PATH) / run_dir.name
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)

    saved_tflite_path = ''
    tflite_src = Path(result['tflite_path']) if result['tflite_path'] else None
    if tflite_src and tflite_src.exists():
        if tflite_src.is_relative_to(run_dir):
            saved_tflite_path = str(dest / tflite_src.relative_to(run_dir))
        else:
            export_dir = dest / 'exports'
            export_dir.mkdir(parents=True, exist_ok=True)
            copied_tflite = export_dir / tflite_src.name
            shutil.copy2(tflite_src, copied_tflite)
            saved_tflite_path = str(copied_tflite)

    print('\nResults saved to Google Drive')
    print(f"  Model size     : {result['model_size']}")
    print(f"  Drive run dir  : {dest}")
    print(f"  Best weights   : {dest / 'weights' / 'best.pt'}")
    print(f"  Last weights   : {dest / 'weights' / 'last.pt'}")
    print(f"  Metrics CSV    : {dest / 'results.csv'}")
    print(f"  Training plot  : {dest / 'results.png'}")
    print(f"  INT8 TFLite    : {saved_tflite_path or '(not copied)'}")
    if result['export_warning']:
        print(f"  Export warning : {result['export_warning']}")
